# Dashboard — Assistente de IA vs. codificação manual

Notebook de visualização do experimento (EPIC #43). O carregamento dos dados
está **pronto** (issue #48, `scripts/dashboard/load_data.py`) e os **gráficos
por RQ** também (issue #49, `scripts/dashboard/plots.py`); as seções de testes
estatísticos seguem como esqueleto, para serem preenchidas pelas issues:

| Seção | Issue |
|---|---|
| Revisão dos dados / outliers | #44 |
| RQ1 — tempo (time-to-green) | #45 (análise) · ~~#49 (gráfico)~~ ✔ |
| RQ2 — taxa de sucesso | #46 (análise) · ~~#49 (gráfico)~~ ✔ |
| RQ3 — métricas estáticas | #47 (análise) · ~~#49 (gráfico)~~ ✔ |
| Montagem final + README | #50 |

Hipóteses, variáveis e testes planejados: [`docs/desenho-experimento.md`](../../docs/desenho-experimento.md).

**Execução:** `pip install -r scripts/dashboard/requirements.txt` e depois
*Run All*. O notebook é reprodutível a partir dos arquivos do repositório —
não há dado digitado aqui. As figuras são renderizadas inline e exportadas em
PNG para `results/figures/` (o mesmo conjunto que
`python scripts/dashboard/plots.py` gera sem abrir o notebook).

## 0. Carga dos dados (issue #48)

`build_dataset()` junta, pela tripla `(participant, kata, treatment)`:

- o dataset consolidado da issue #44 (`dados/consolidado.csv`), se existir;
- `results/timing.json` — tempo, censura e contagem de testes (RQ1/RQ2);
- `results/static_metrics.csv` — LOC, complexidade, duplicação, MI (RQ3).

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# raiz do repo a partir da localizacao deste notebook (sem caminho absoluto)
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "scripts" / "dashboard" / "load_data.py").exists() \
        and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "scripts" / "dashboard"))

from load_data import (  # noqa: E402
    DATA_DICTIONARY,
    METRIC_COLUMNS,
    build_dataset,
    build_paired_frame,
    build_participant_summary,
    descriptives_by_treatment,
    describe_dataset,
    success_rate_by_treatment,
    validate_dataset,
)
from plots import (  # noqa: E402  (issue #49)
    RQ3_METRICS,
    export_all,
    figure_rq1,
    figure_rq2,
    figure_rq3_metric,
    figure_rq3_overview,
    save_figure,
)

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)

FIG_DIR = REPO_ROOT / "results" / "figures"   # destino das figuras (issue #49)
FIG_DIR.mkdir(parents=True, exist_ok=True)

df = build_dataset(repo_root=REPO_ROOT)
print("fontes:", df.attrs.get("sources"))
df.head(len(df))

### 0.1 Dicionário de dados

Uma linha por trial. Colunas, tipos e a que RQ cada uma serve:

In [ ]:
print(describe_dataset(df))

### 0.2 Integridade da junção

`validate_dataset` **não** exclui nada — só aponta o que precisa de decisão
documentada na issue #44 (trials em apenas uma das fontes, kata repetida por
integrante, integrante sem par completo, tempo acima do time-box).

In [ ]:
avisos = validate_dataset(df)
print("\n".join(f"- {a}" for a in avisos) if avisos else "nenhum aviso de integridade")

## 1. Visão geral da amostra

O desenho é crossover within-subject: a unidade de análise é o **integrante**,
com observações pareadas `manual` x `ai` (desenho, §5).

In [ ]:
visao = (df.pivot_table(index="participant", columns="treatment", values="trial_id",
                        aggfunc="count", observed=True)
           .fillna(0).astype(int)
           .rename(columns={"manual": "trials_manual", "ai": "trials_ai"}))
display(visao)
display(success_rate_by_treatment(df))

## 2. RQ1 — Tempo até o verde (*time-to-green*)

- **H1₀:** `θ(time_to_green_ia) = θ(time_to_green_manual)` (bicaudal, α = 0,05).
- Variável: `time_to_green_min`, **censurada à direita** em 35 min
  (`censored` / `event`).
- Análise planejada (issue #45): Kaplan-Meier + log-rank (ou RMST em 35 min)
  como primária; Wilcoxon pareado como secundária, declaradamente conservadora.

In [ ]:
display(descriptives_by_treatment(df, ["time_to_green_min"]))
display(df.loc[:, ["trial_id", "treatment", "time_to_green_min", "censored", "event", "status"]])
print("censurados por tratamento:")
print(df.groupby("treatment", observed=True)["censored"].apply(lambda s: int(s.fillna(False).sum())))

In [ ]:
# Issue #49 - RQ1: boxplot de time_to_green_min por tratamento (mediana/IQR),
# com os trials censurados sinalizados no teto do time-box, ao lado do grafico
# de pares manual -> ai de cada integrante. O backend inline renderiza a figura
# no fim da celula; save_figure grava o PNG para o Relatorio Final.
fig_rq1 = figure_rq1(df)
save_figure(fig_rq1, FIG_DIR / "rq1_time_to_green.png")

# TODO(#45): Kaplan-Meier por tratamento + log-rank (lifelines), usando
#            time_to_green_min como duracao e `event` como indicador.
#            Reportar tamanho de efeito (Cliff's delta) e n de censurados.

## 3. RQ2 — Defeitos / taxa de sucesso

- **Primário:** taxa de sucesso (proporção de trials que atingem o verde dentro
  do time-box) — binária pareada, **McNemar** (issue #46).
- **Secundário:** `pct_tests_passing`, que por construção só varia entre os
  trials censurados (ver o alerta de operacionalização no desenho, §3).

In [ ]:
display(success_rate_by_treatment(df))
display(build_participant_summary(df)[["participant", "treatment", "n_trials",
                                        "n_success", "n_censored", "success_rate",
                                        "pct_tests_passing"]])

In [ ]:
# Issue #49 - RQ2: barras da taxa de sucesso por tratamento (variavel primaria,
# com n e censurados por barra) e boxplot de pct_tests_passing (secundaria, que
# so varia entre os trials censurados).
fig_rq2 = figure_rq2(df)
save_figure(fig_rq2, FIG_DIR / "rq2_taxa_sucesso.png")

# TODO(#46): teste de McNemar sobre os pares por integrante (statsmodels), mais
#            a leitura secundaria em pct_tests_passing. Registrar que, com todos
#            os trials verdes, o teste nao tem variabilidade para detectar efeito.

## 4. RQ3 — Qualidade estrutural do código

Família de três hipóteses (correção de Holm-Bonferroni dentro da família):

- **H3a** `cc_avg` — complexidade ciclomática média por função;
- **H3b** `duplication_pct` — percentual de linhas duplicadas;
- **H3c** `loc` — tamanho da solução.

Exploratórias, fora da correção: `cc_total`, `mi_avg`, `sloc`.

In [ ]:
display(descriptives_by_treatment(df, ["cc_avg", "duplication_pct", "loc", "sloc",
                                        "cc_total", "mi_avg"]))
display(build_paired_frame(df))

In [ ]:
# Issue #49 - RQ3: painel com as tres metricas da familia (H3a cc_avg,
# H3b duplication_pct, H3c loc) e, em seguida, uma figura por metrica com o
# boxplot (mediana/IQR) ao lado dos pares por integrante.
fig_rq3 = figure_rq3_overview(df)
save_figure(fig_rq3, FIG_DIR / "rq3_estrutura.png")

for metrica in RQ3_METRICS:
    fig_metrica = figure_rq3_metric(df, metrica)
    save_figure(fig_metrica, FIG_DIR / f"rq3_{metrica}.png")

# TODO(#47): Wilcoxon signed-rank pareado por integrante nas tres metricas,
#            Shapiro-Wilk nas diferencas (teste t pareado em paralelo quando
#            couber), Holm-Bonferroni na familia e Cliff's delta como efeito.
#            Discutir o acoplamento entre duplication_pct e loc (desenho, §8).

## 5. Síntese para o Relatório Final

Preencher na issue #50, a partir dos resultados acima: uma linha por RQ com
mediana por tratamento, p-valor, tamanho de efeito e a leitura (incluindo o
aviso de baixo poder estatístico do desenho, §7).

In [ ]:
# Figuras exportadas por este notebook (issue #49), na ordem em que entram no
# relatorio. `export_all(df, FIG_DIR)` regenera todas de uma vez, fora do
# notebook -- e o que `python scripts/dashboard/plots.py` faz.
# TODO(#50): tabela-resumo (uma linha por RQ: mediana por tratamento, p-valor,
#            tamanho de efeito e leitura), referenciando estas figuras.
sorted(p.name for p in FIG_DIR.glob("*.png"))